<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;"> 
    <h1 style="margin-top: 0; font-size: 28px;">PURPLE NOTEBOOK </h1>
    <p style="margin-bottom: 0;">PACE BGC to abundance based PCC in jupyter notebook, IOCCG SLS, by Ivona</p>

<div style="background-color: #9575CD; padding: 20px; border-radius: 8px; color: #FFFFFF;">
Here we will try to reproduce the figures Ivona used in her Phytoplankton from space talk. First we will load the specific file for the Gulf of Maine, BGC suite. If you want to use this code on other areas just modify the query as seen above. 

PACE BGC suite has several products, depending on the version/reprocessing of the data, number of products may vary as we keep adding products. For example, current version [version 3.2](https://www.earthdata.nasa.gov/data/catalog/ob-cloud-pace-oci-l2-bgc-nrt-3.2), has PIC product that was not in previous versions. 

Since v3.2, BGC suite has the same geophyiscal parameters in L2 and L3 (L3 doesn't have uncertanties products). 

To run this notebook you need an account with Earthdata. You can create one by navigating to the [Earthdata Login Registration page](https://urs.earthdata.nasa.gov/) and selecting "Register for a profile" to provide your basic information.

As it is written, code needs 2GB of memory to run. 

</div>

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
We begin by adding the external packages to your enviroment. If you are running this in google colab, you will have everything except for earthaccess and cartopy. If you are at cryocloud, you should have cartopy and earthaccess in your enviroment so comment this out. If you are somewhere elsewhere, just uncoment the whole line and run it all.

In [ ]:
!pip install earthaccess cartopy #matplotlib numpy xarray

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
Let us then load the libraries you will need for this notebook. 

In [ ]:
import earthaccess
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
from xarray.backends.api import open_datatree
from matplotlib.colors import LogNorm  # Imported LogNorm for log10 scale


# Ensure inline plotting works inside the notebook
%matplotlib inline

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
Then we will load the specific file for the Gulf of Maine, BGC suite, as that is where our chlorophyll product is hiding. We will also define the bbox, box of interest that will be used for plotting. 

In [ ]:
bbox = (-77, 30, -60, 42)

This is assuming that you have previously logged into `earthaccess` on this platform and that it has remembered your password and login. If not, please go to this [notebook](https://nasa.github.io/oceandata-notebooks/notebooks/oci/oci_data_access.html) and follow the directions.


In [ ]:
auth = earthaccess.login(persist=True)

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
We are streaming data only, not downloading. Once data is loaded into the data tree, you can explore the datatree in the structure printout below, and see all the pretty parameters you have. 

In [ ]:
results = earthaccess.search_data(
    concept_id='G4184455498-OB_CLOUD',
    granule_name='PACE_OCI.20260505T173406.L2.OC_BGC.V3_2.NRT.nc'
)

# Stream the file directly into memory
file_handle = earthaccess.open(results)
datatree = open_datatree(file_handle[0])
datatree

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
Now we flatten the data to play with it. 

In [ ]:
ds = xr.merge(datatree.to_dict().values())
print(ds)

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
This is a little function that is following the Brewin et al, 2010 paper, where we can decompose the chlorophyll product in three size classes, micro, nano, and picophytoplankton.

In [ ]:
def brewin_three_component_model(chla):
    """
    Calculates the fractional concentrations of phytoplankton size classes 
    based on the Brewin et al. (2010) model for the Atlantic Ocean.
    
    Handles 2D numpy arrays with NaN values automatically.
    """
    # Convert input to a numpy array to ensure consistency
    C = np.array(chla, dtype=float)
    
    # Brewin et al. (2010) parameters optimized for the Atlantic
    C_m_pico = 0.13      # Asymptotic maximum for pico
    S_pico = 5.15        # Initial slope for pico
    C_m_combined = 0.77  # Asymptotic maximum for pico + nano
    S_combined = 1.22    # Initial slope for pico + nano
    
    # Compute absolute concentrations (mg/m^3)
    C_pico = C_m_pico * (1.0 - np.exp(-S_pico * C))
    C_pico_nano = C_m_combined * (1.0 - np.exp(-S_combined * C))
    
    C_nano = C_pico_nano - C_pico
    C_micro = C - C_pico_nano
    
    # Maintain physical constraints (no negative biomass)
    C_micro = np.maximum(C_micro, 0.0)
    C_nano = np.maximum(C_nano, 0.0)
    C_pico = np.maximum(C_pico, 0.0)
    
    # Safely calculate fractions (avoiding division by zero)
    total_safe = np.where((C == 0) | np.isnan(C), 1e-6, C)
    F_pico = C_pico / total_safe
    F_nano = C_nano / total_safe
    F_micro = C_micro / total_safe
    
    # Restore NaN values where the original data had no coverage (e.g., land masks)
    nan_mask = np.isnan(C)
    for arr in [C_pico, C_nano, C_micro, F_pico, F_nano, F_micro]:
        arr[nan_mask] = np.nan
        
    return {
        "pico_abs": C_pico, "nano_abs": C_nano, "micro_abs": C_micro,
        "pico_frac": F_pico, "nano_frac": F_nano, "micro_frac": F_micro
    }


<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
OK :) now run the magical function here! It will save the output into the .nc file so you can have it forever and ever. First, extract the chlorophyll-a variable mapping to (number_of_lines, pixels_per_line), and then process the 2D array matrix through the Brewin et al. 2010 model

In [ ]:
chl_data = ds["chlor_a"]
results = brewin_three_component_model(chl_data.values)

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
This way you can map the output arrays directly back to the dataset using PACE dimensions, and add metadata attributes (like units and definition)

In [ ]:
spatial_dims = ("number_of_lines", "pixels_per_line")
ds["pico_fraction"] = (spatial_dims, results["pico_frac"] * 100) # Convert to %
ds["nano_fraction"] = (spatial_dims, results["nano_frac"] * 100)
ds["micro_fraction"] = (spatial_dims, results["micro_frac"] * 100)

ds["pico_fraction"].attrs = {"units": "%", "long_name": "Picophytoplankton Fraction (<2 µm)"}
ds["nano_fraction"].attrs = {"units": "%", "long_name": "Nanophytoplankton Fraction (2-20 µm)"}
ds["micro_fraction"].attrs = {"units": "%", "long_name": "Microphytoplankton Fraction (>20 µm)"}

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
Fabulous! Now let us make cute little plots. First Chlorophyll. 
Note - all my code automatically saves the figure files as publication style (higer size) plots. Stop now here, and comment out second to last line of the code if you don't want that. 

In [ ]:
variables = ["chlor_a"]

# ---------------------------------------------------------
# Clean lon/lat once (avoids blank-map issue from fill values like -9999)
# ---------------------------------------------------------
lon_clean = ds["longitude"].values.astype(float)
lat_clean = ds["latitude"].values.astype(float)
lon_clean[(lon_clean < -180) | (lon_clean > 180)] = np.nan
lat_clean[(lat_clean < -90) | (lat_clean > 90)] = np.nan

# Reorder bbox for Cartopy's set_extent: [lon_min, lon_max, lat_min, lat_max], we defined the bbox above
extent = [bbox[0], bbox[2], bbox[1], bbox[3]]

crs = ccrs.PlateCarree()

# Note: Since variables is length 1, axs might not be an array if using plt.subplots(1, 1)
# To safely iterate, we can ensure it's iterable by using squeeze=False or reshaping
fig, axs = plt.subplots(1, 1, figsize=(6, 5), subplot_kw={"projection": crs})
if not isinstance(axs, np.ndarray):
    axs = np.array([axs])

im = None  # keep reference for shared colorbar later in the code

# Note this code can be modified to plot multiple parameters are the same time hence the loop. 
for i, ax in enumerate(axs.flat):
    if i < len(variables):
        var = variables[i]

        data = ds[var].values.astype(float)

        # Mask out-of-range fill values in the data itself.
        # Log10 requires strictly positive values, so we mask data <= 0 as well.
        data[(data <= 0) | (data > 10000)] = np.nan

        lon = lon_clean
        lat = lat_clean

        # Fixed color scale limits (must be > 0 for LogNorm)
        vmin, vmax = 0.01, 100

        # Add land features
        ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', linewidth=0.5)
        ax.add_feature(cfeature.COASTLINE, linewidth=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--', alpha=0.5)
        ax.add_feature(cfeature.LAKES, facecolor='lightgray', alpha=0.7)

        im = ax.pcolormesh(
            lon, lat, data,
            cmap="viridis",
            shading="auto",
            zorder=10,
            norm=LogNorm(vmin=vmin, vmax=vmax), # Apply Log10 Normalization here
            transform=ccrs.PlateCarree()
        )

        ax.gridlines(
            draw_labels=["left", "bottom"],
            linewidth=0.5,
            color="gray",
            alpha=0.5,
            linestyle="--",
        )

        ax.set_title(var.upper())

        # Set map extent using Cartopy-native method
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    else:
        ax.set_visible(False)

plt.subplots_adjust(hspace=0.05, right=0.82)

cbar_ax = fig.add_axes([0.85, 0.15, 0.03, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation="vertical")
cbar.set_label("Chlorophyll a (mg $m^{-3}$)")

# Save the figure - filename to reflect log10 scale
plt.savefig('chlorophyll_concentration_log10.png', dpi=250, bbox_inches='tight')

plt.show()

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">
And now, Brewin's three size classes! Feel free to change the colorscale or whatever you want. I just used the same as chlorophyll. 

In [ ]:
# ---------------------------------------------------------
# Variables to plot
# ---------------------------------------------------------
variables = ["micro_fraction", "nano_fraction", "pico_fraction"]

# ---------------------------------------------------------
# Clean lon/lat once (avoids blank-map issue from fill values like -9999)
# ---------------------------------------------------------
lon_clean = ds["longitude"].values.astype(float)
lat_clean = ds["latitude"].values.astype(float)
lon_clean[(lon_clean < -180) | (lon_clean > 180)] = np.nan
lat_clean[(lat_clean < -90) | (lat_clean > 90)] = np.nan

# Reorder bbox for Cartopy's set_extent: [lon_min, lon_max, lat_min, lat_max]
extent = [bbox[0], bbox[2], bbox[1], bbox[3]]

crs = ccrs.PlateCarree()

fig, axs = plt.subplots(3, 1, figsize=(6, 15), subplot_kw={"projection": crs})

im = None  # keep reference for shared colorbar

# Iterate over all 3 subplots
for i, ax in enumerate(axs.flat):
    if i < len(variables):
        var = variables[i]

        data = ds[var].values.astype(float)

        # Mask out-of-range fill values in the data itself (fractions should be 0-100%)
        data[(data < 0) | (data > 100)] = np.nan

        lon = lon_clean
        lat = lat_clean

        # Fixed color scale for all three plots
        vmin, vmax = 0, 100

        # Add land features
        ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', linewidth=0.5)
        ax.add_feature(cfeature.COASTLINE, linewidth=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--', alpha=0.5)
        ax.add_feature(cfeature.LAKES, facecolor='lightgray', alpha=0.7)

        im = ax.pcolormesh(
            lon, lat, data,
            cmap="viridis",
            shading="auto",
            zorder=10,
            vmin=vmin,
            vmax=vmax,
            transform=ccrs.PlateCarree()
        )

        ax.gridlines(
            draw_labels=["left", "bottom"],
            linewidth=0.5,
            color="gray",
            alpha=0.5,
            linestyle="--",
        )

        ax.set_title(var.upper())

        # Set map extent using Cartopy-native method
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    else:
        ax.set_visible(False)

# Reduce vertical spacing between the stacked subplots
# Also leave room on the right for the shared colorbar
plt.subplots_adjust(hspace=0.05, right=0.82)

# Single shared colorbar for all three subplots, fixed 0-100%
cbar_ax = fig.add_axes([0.85, 0.15, 0.03, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation="vertical")
cbar.set_label("Phytoplankton Size Fraction (%)")

# Save the figure - optional
plt.savefig('Brewin_size_fractions_map_linear.png', dpi=250, bbox_inches='tight')

plt.show()

<div style="background-color: #9575CD; padding: 20px; border-radius: 8px; color: #FFFFFF;">

Coolio. Now it's your turn to play! :) 
    
Here are a couple of ideas:

1. Can you plot other parameters and see how they vary in this area? For example, the phytoplankton carbon product tells you how much phytoplankton carbon there is in the area. However, the current NASA's [phytoplankton carbon algorithm](https://oceancolor.gsfc.nasa.gov/files/atbd/legacy/atbd-obdaac-phytoplankton-carbon-concentration.pdf) is just a linear relationship with backscattering, so it might be picking up other things. Can you plot the PIC product and see how does it vary in this scene? Explore the covariability of these products. 

2. If you are adventurous enough, check out the chlorophyll uncertainty and try to propagate it through the Brewin algorithm? For ideas on how to do this, check out the appendix of [McKinna et al., 2019](https://doi.org/10.3389/feart.2019.00176).

3. The slope of the backscattering coefficient is a proxy for many things in the ocean, one of which is size. Can you compare the backscattering slope (available in the [L2 IOP](https://www.earthdata.nasa.gov/data/catalog/ob-cloud-pace-oci-l2-iop-nrt-3.2) suite) with the suggested distribution of phytoplankton from Brewin's algorithm? Before publishing your amazing results, review the assumptions of both algorithms and identify their shortcomings.

4. Do you want to code in another PCC algorithm that derives size distribution? Maybe try [Hirata et al., 2011](https://doi.org/10.5194/bg-8-311-2011) and compare the outputs. 


</div>